# Collect ROS elevation-map data

Run this on a **hosted Colab NVIDIA GPU**. The setup cell installs ROS 2 Jazzy,
the project's ROS dependencies, CuPy and the MuJoCo LiDAR simulator directly
into Colab's temporary Ubuntu 24.04 runtime. It may take a while and needs
substantial disk space. The notebook records the real ROS mapping pipeline;
the policy receives the true-terrain scan so differences in mapping do not
change the robot trajectory. No training takes place.

Run cells in order. The final cell writes one `.tar.gz` file to
`MyDrive/humanoid/ros_data/`; share that file for analysis. Use a branch that
already contains this notebook and `training/scripts/setup_ros_colab.sh`.


In [ ]:
#@title 1. Authorize GitHub and clone the branch
REPO = "university-of-toronto-humanoid-robotics/humanoid" #@param {type:"string"}
BRANCH = "kartik/rpl" #@param {type:"string"}

from pathlib import Path
import os, shutil, subprocess, sys
from IPython.display import HTML, display

REPO_DIR = Path("/content/humanoid")
if shutil.which("gh") is None:
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "gh"], check=True)
git_env = {**os.environ, "GH_BROWSER": "echo", "GH_PROMPT_DISABLED": "1", "GIT_TERMINAL_PROMPT": "0"}
git_env.pop("GH_TOKEN", None)
git_env.pop("GITHUB_TOKEN", None)
if subprocess.run(["gh", "auth", "status", "--hostname", "github.com"], env=git_env,
                  stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL).returncode:
    display(HTML('<a href="https://github.com/login/device" target="_blank">Open GitHub authorization</a><p>Enter the code below; this cell waits for approval.</p>'))
    login = subprocess.Popen(["gh", "auth", "login", "--web", "--hostname", "github.com",
                              "--git-protocol", "https"], env=git_env,
                             stdin=subprocess.DEVNULL, stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True)
    for line in login.stdout:
        print(line, end="", flush=True)
    if login.wait():
        raise RuntimeError("GitHub login failed. Check the output above.")
subprocess.run(["gh", "auth", "setup-git", "--hostname", "github.com"], env=git_env, check=True)
url = f"https://github.com/{REPO}.git"
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, url, str(REPO_DIR)], env=git_env, check=True)
else:
    origin = subprocess.check_output(["git", "-C", str(REPO_DIR), "remote", "get-url", "origin"], text=True).strip()
    assert origin.lower().removesuffix(".git") == url.lower().removesuffix(".git"), "Existing checkout uses a different repository; use a fresh runtime."
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], env=git_env, check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "switch", "-C", BRANCH, f"origin/{BRANCH}"], env=git_env, check=True)
assert (REPO_DIR / "training/scripts/setup_ros_colab.sh").is_file(), "Push this notebook and setup script to the selected branch first."
print("Repository revision:", subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"], text=True).strip())


In [ ]:
# 2. Mount Drive and check the hosted GPU runtime before the long install.
from google.colab import drive
import platform
drive.mount("/content/drive")
OUTPUT_DIR = Path("/content/drive/MyDrive/humanoid/ros_data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
os_release = Path("/etc/os-release").read_text()
assert "ID=ubuntu" in os_release and 'VERSION_ID="24.04"' in os_release, "Use a hosted Colab Ubuntu 24.04 runtime; this ROS Jazzy setup will not install on another Ubuntu release."
assert platform.machine() == "x86_64", "ROS setup needs an x86_64 runtime."
assert os.geteuid() == 0, "ROS package installation needs Colab's root runtime."
subprocess.run(["nvidia-smi"], check=True)
free_gb = shutil.disk_usage("/content").free / 1e9
assert free_gb >= 12, f"Only {free_gb:.1f} GB disk free; allow at least 12 GB for ROS, the build and bag."
checkpoint = REPO_DIR / "training/models/old/best_g1_rough_heightmap/model_5996.pt"
assert checkpoint.is_file(), f"Tracked rough-heightmap checkpoint missing: {checkpoint}"
print(f"GPU, checkpoint and Drive ready. Free disk: {free_gb:.1f} GB")


In [ ]:
# 3. Install ROS Jazzy and build this repository's ROS workspace.
# The helper uses /usr/bin/python3.12 for ROS; the Colab notebook kernel keeps its own Python.
setup = REPO_DIR / "training/scripts/setup_ros_colab.sh"
process = subprocess.Popen(["bash", str(setup), str(REPO_DIR)],
                           stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                           text=True, bufsize=1)
try:
    for line in process.stdout:
        print(line, end="", flush=True)
    result = process.wait()
except KeyboardInterrupt:
    process.terminate()
    process.wait()
    raise
if result:
    raise subprocess.CalledProcessError(result, process.args)
print("ROS build complete.")


In [ ]:
#@title 4. Run the ROS simulator and record mapping topics
SCENARIOS = "room_stairs_heightmap,course_ramps_heightmap" #@param {type:"string"}
INCLUDE_RAW_LIDAR = False #@param {type:"boolean"}

import json, shlex, signal, time
from datetime import datetime, timezone

allowed = {"room_stairs_heightmap", "course_ramps_heightmap"}
selected = [name.strip() for name in SCENARIOS.split(",") if name.strip()]
assert selected and set(selected) <= allowed, f"Choose one or both of: {', '.join(sorted(allowed))}"
RUN_DIR = Path("/content/ros_capture") / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RUN_DIR.mkdir(parents=True)
ROS_ENV = os.environ.copy()
ROS_ENV.pop("PYTHONPATH", None)
ROS_ENV.update({
    "PATH": "/usr/bin:/bin:" + os.environ.get("PATH", ""),
    "PYTHONNOUSERSITE": "1", "ROS_DOMAIN_ID": "42", "ROS_AUTOMATIC_DISCOVERY_RANGE": "LOCALHOST",
    "RMW_IMPLEMENTATION": "rmw_cyclonedds_cpp", "MUJOCO_GL": "egl",
    "LD_LIBRARY_PATH": "/usr/local/lib/python3.12/dist-packages/nvidia/cuda_nvrtc/lib:/usr/local/lib/python3.12/dist-packages/nvidia/cuda_runtime/lib:" + os.environ.get("LD_LIBRARY_PATH", ""),
})
source = f"source /opt/ros/jazzy/setup.bash && source {shlex.quote(str(REPO_DIR / 'ros2_ws/install/setup.bash'))} && exec \"$@\""
def ros_command(argv, **kwargs):
    return subprocess.Popen(["bash", "-c", source, "ros", *map(str, argv)],
                            cwd=REPO_DIR, env=ROS_ENV, **kwargs)
def stop_group(process):
    if process and process.poll() is None:
        os.killpg(process.pid, signal.SIGINT)
        try:
            process.wait(timeout=25)
        except subprocess.TimeoutExpired:
            os.killpg(process.pid, signal.SIGTERM)
            process.wait(timeout=10)

topics = ["/clock", "/tf", "/tf_static", "/joint_states", "/lowstate",
          "/ground_truth/odom", "/ground_truth/height_scan", "/height_scan",
          "/height_scan/observed_share", "/height_scan/feet_share",
          "/elevation_mapping_node/elevation_map_filter"]
if INCLUDE_RAW_LIDAR:
    topics += ["/livox/lidar", "/livox/imu", "/cloud_self_filtered"]

for scenario in selected:
    folder = RUN_DIR / scenario
    folder.mkdir()
    bag_path = folder / "bag"
    print(f"Starting ROS bag for {scenario}...", flush=True)
    with (folder / "bag.log").open("w") as bag_log, (folder / "launch.log").open("w") as launch_log:
        bag = launch = None
        try:
            bag = ros_command(["ros2", "bag", "record", "-o", bag_path, *topics],
                              stdout=bag_log, stderr=subprocess.STDOUT, start_new_session=True)
            time.sleep(4)
            if bag.poll() is not None:
                raise RuntimeError(f"ROS bag exited early; see {folder / 'bag.log'}")
            launch = ros_command(["ros2", "launch", "bringup", "g1.launch.py",
                                  f"scenario:={scenario}", "scan:=true_terrain",
                                  f"checkpoint:={checkpoint}", "viewer:=false",
                                  "rviz:=false", "localize:=false"],
                                 stdout=launch_log, stderr=subprocess.STDOUT,
                                 start_new_session=True)
            started = time.monotonic()
            while launch.poll() is None:
                print(f"{scenario}: {time.monotonic() - started:.0f}s elapsed", flush=True)
                if time.monotonic() - started > 1800:
                    raise TimeoutError(f"{scenario} exceeded 30 minutes; see {folder / 'launch.log'}")
                time.sleep(15)
            if launch.returncode:
                print((folder / "launch.log").read_text(errors="replace")[-5000:])
                raise RuntimeError(f"ROS launch exited {launch.returncode}; see {folder / 'launch.log'}")
        finally:
            stop_group(launch)
            stop_group(bag)
    launch_text = (folder / "launch.log").read_text(errors="replace")
    if "Elevation-map scan vs the true one over" not in launch_text:
        print(launch_text[-5000:])
        raise RuntimeError(f"No live ROS height scan was reported for {scenario}; see launch.log")
    report = next((line for line in launch_text.splitlines() if "Elevation-map scan vs the true one over" in line), "")
    info = ros_command(["ros2", "bag", "info", bag_path], stdout=subprocess.PIPE,
                       stderr=subprocess.STDOUT, text=True)
    info_text = info.communicate()[0]
    (folder / "bag_info.txt").write_text(info_text)
    if info.returncode:
        raise RuntimeError(f"Cannot read recorded bag: {info_text}")
    for topic in ("/height_scan", "/ground_truth/height_scan",
                  "/elevation_mapping_node/elevation_map_filter"):
        if topic not in info_text:
            raise RuntimeError(f"{topic} was not recorded for {scenario}; see bag_info.txt and launch.log")
    print(report, flush=True)
    print(f"Recorded {scenario} in {folder}", flush=True)

manifest = {"repo": REPO, "branch": BRANCH,
            "commit": subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip(),
            "scenarios": selected, "topics": topics, "policy_scan_source": "true_terrain",
            "checkpoint": str(checkpoint.relative_to(REPO_DIR)),
            "gpu": subprocess.check_output(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True).strip()}
(RUN_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2))
print("Capture complete:", RUN_DIR)


In [ ]:
# 5. Bundle the bags and logs into one Drive file to share.
import tarfile
assert RUN_DIR.is_dir(), "Run cell 4 first."
archive = Path("/content") / f"{RUN_DIR.name}_ros_mapping.tar.gz"
with tarfile.open(archive, "w:gz") as tar:
    tar.add(RUN_DIR, arcname=RUN_DIR.name)
destination = OUTPUT_DIR / archive.name
shutil.copy2(archive, destination)
print(f"Saved: {destination} ({destination.stat().st_size / 1e6:.1f} MB)")
print("Share this archive for the paired ROS-versus-student map analysis.")
